# 01 — Data Understanding and Preprocessing

**Goal:** understand the VisDrone video data before training any detector, and collect the facts
that drive later choices (validation split, input resolution, anchors, augmentation, ignored regions).

**Contents**
1. Dataset structure and file formats
2. Loading annotations, class mapping and ignored regions
3. Frame resolution and video length
4. Class distribution
5. Object sizes
6. Scene density
7. Ground-truth visualisation (conversion check)
8. Summary: key observations and decisions

All figures are saved to `<work>/figs/` (Kaggle: `/kaggle/working/figs`, Colab: `MyDrive/vdet/figs`, laptop: `~/vdet_work/figs`).

## 0. Setup
Works on **Kaggle** (competition added as input), **Google Colab** (`kaggle.json` in the root of
Google Drive) and **a laptop** (`kaggle.json` in Downloads or `~/.kaggle/`). On Colab and on a laptop
the data is downloaded once with the Kaggle API. No GPU is needed for this notebook.

In [1]:
REPO = "https://github.com/LidaSokha03/AI_HW2.git"   # shared code is in src/ of this repository

In [ ]:
import os
import sys
import json
import subprocess

if "google.colab" in sys.modules:  # Colab: keep results on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle>=2.0", "numpy", "pandas", "matplotlib", "pillow"], check=True)

# Shared code (src/vdet.py): local copy if present, otherwise cloned from GitHub
CANDIDATES = ["src", "../src", ".", "notebooks"]
SRC = os.environ.get("VDET_SRC") or next((p for p in CANDIDATES if os.path.exists(f"{p}/vdet.py")), None)
if SRC is None:
    if os.path.exists("/tmp/AI_HW2"):
        subprocess.run(["git", "-C", "/tmp/AI_HW2", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/tmp/AI_HW2"], check=True)
    SRC = next(f"/tmp/AI_HW2/{p}" for p in ["src", "notebooks"] if os.path.exists(f"/tmp/AI_HW2/{p}/vdet.py"))
sys.path.insert(0, os.path.abspath(SRC))

import vdet

P = vdet.get_paths()
COMP = vdet.get_comp(P)            # original competition data (downloaded once on Colab / laptop)
WORK = P["work"]
FIGS = f"{WORK}/figs"
os.makedirs(FIGS, exist_ok=True)
print("platform:", P["platform"], "| data:", COMP, "| outputs:", WORK)


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

CLASS_NAMES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
COLORS = plt.cm.tab10.colors

## 1. Dataset structure and file formats

### 1.1 Folder structure
Only the first 3 levels are listed, so we do not walk through tens of thousands of frames.

In [ ]:
def show(path, depth=0, max_depth=3):
    """Print a short tree: number of sub-folders and files, plus 3 example files per folder."""
    try:
        entries = sorted(os.listdir(path))
    except NotADirectoryError:
        return
    dirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    files = [e for e in entries if not os.path.isdir(os.path.join(path, e))]
    print("  " * depth + os.path.basename(path) + f"/  ({len(dirs)} dirs, {len(files)} files)")
    for f in files[:3]:
        size = os.path.getsize(os.path.join(path, f)) / 1e6
        print("  " * (depth + 1) + f"{f}  ({size:.2f} MB)")
    if depth < max_depth:
        for d in dirs[:3]:
            show(os.path.join(path, d), depth + 1, max_depth)

show(COMP)

### 1.2 Splits and the Ultralytics config
Each split has one sub-folder per video. `train` and `val` have `images/`, `labels/` (YOLO format)
and `ignore/` (ignored regions); `test` has only images.

In [ ]:
for split in ["train", "val", "test"]:
    for d in sorted(os.listdir(f"{COMP}/{split}")):
        p = f"{COMP}/{split}/{d}"
        if os.path.isdir(p):
            items = sorted(os.listdir(p))
            print(f"{split}/{d}/: {len(items)} videos, e.g. {items[:3]}")

print("\n--- train/data.yaml ---")
print(open(f"{COMP}/train/data.yaml").read())

### 1.3 Test frames
`test_images.csv` lists every test frame. Its `image_id` is the key used in the submission file.

In [ ]:
test_df = pd.read_csv(f"{COMP}/test_images.csv")
print("test frames:", len(test_df), "| clips:", test_df["clip"].nunique())
test_df.head()

### 1.4 COCO annotation format
We look at the smaller `val` file. Boxes are `[x, y, width, height]` in pixels (top-left corner).
Besides the class, every box has `iscrowd`, `truncation`, `occlusion`, `track_id` and the original
VisDrone class `orig_category_id`.

In [ ]:
with open(f"{COMP}/val/annotations_coco.json") as f:
    coco_val = json.load(f)

print("keys:", list(coco_val.keys()))
print("categories:", coco_val["categories"])
print("images[0]:", coco_val["images"][0])
print("annotations[0]:", coco_val["annotations"][0])
del coco_val  # the full files are loaded in section 2

### 1.5 YOLO labels and ignored-region files
Both are one `.txt` file per frame. Labels: `class cx cy w h` (normalised to 0..1).
Ignored regions: `cx cy w h` (same format, no class).

In [ ]:
def peek(path, n_lines=4):
    """Show how many files a folder has and the first lines of the first file."""
    files = sorted(os.listdir(path))
    print(f"{path.replace(COMP, '')}/ -> {len(files)} files, e.g. {files[:3]}")
    with open(f"{path}/{files[0]}") as f:
        print("".join(f.readlines()[:n_lines]))

vid = sorted(os.listdir(f"{COMP}/train/ignore"))[0]
peek(f"{COMP}/train/ignore/{vid}")
peek(f"{COMP}/train/labels/{vid}")

## 2. Loading annotations

### 2.1 COCO files to pandas tables
One table with frames (`imgs_all`) and one with boxes (`anns_all`) for train and val together.

In [ ]:
def load_coco(split):
    """Read a COCO json and return (frames table, boxes table) with video and size info attached."""
    with open(f"{COMP}/{split}/annotations_coco.json") as f:
        coco = json.load(f)
    imgs = pd.DataFrame(coco["images"])
    anns = pd.DataFrame(coco["annotations"])
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    anns["category"] = anns["category_id"].map(cats)
    anns[["x", "y", "w", "h"]] = pd.DataFrame(anns["bbox"].tolist(), index=anns.index)
    anns = anns.merge(imgs[["id", "video", "width", "height", "frame_id"]],
                      left_on="image_id", right_on="id", suffixes=("", "_img"))
    imgs["split"], anns["split"] = split, split
    return imgs, anns

imgs_tr, anns_tr = load_coco("train")
imgs_va, anns_va = load_coco("val")
imgs_all = pd.concat([imgs_tr, imgs_va], ignore_index=True)
anns_all = pd.concat([anns_tr, anns_va], ignore_index=True)
del imgs_tr, anns_tr, imgs_va, anns_va

print("frames:", imgs_all.groupby("split").size().to_dict())
print("videos:", imgs_all.groupby("split")["video"].nunique().to_dict())
print("boxes: ", anns_all.groupby("split").size().to_dict())
print("iscrowd:", anns_all["iscrowd"].value_counts().to_dict())

### 2.2 Class mapping and ignored regions
The original VisDrone data has 10 object classes plus "ignored region" (0) and "others" (11).
The table shows how they were mapped to the 8 target classes.

In [ ]:
VISDRONE = {0: "ignored region", 1: "pedestrian", 2: "people", 3: "bicycle", 4: "car", 5: "van",
            6: "truck", 7: "tricycle", 8: "awning-tricycle", 9: "bus", 10: "motor", 11: "others"}

def target_classes(s):
    return s.iloc[0] if s.nunique() == 1 else f"all {s.nunique()} classes"

mapping = (anns_all.groupby(["orig_category_id", "iscrowd"])
           .agg(n_boxes=("category", "size"), target_class=("category", target_classes)))
mapping.insert(0, "visdrone_name", [VISDRONE[i] for i in mapping.index.get_level_values(0)])
mapping

"Ignored region" (0) and "others" (11) are stored as `iscrowd=1` boxes, **copied once for each of
the 8 classes**. This is for evaluation: `pycocotools` does not count detections of any class inside
these boxes as false positives. They are **not** objects, so we separate them from the real objects.

In [ ]:
# Real objects (training targets) vs ignore boxes (used only for evaluation).
# Every ignore region is copied for all 8 classes, so one copy (category_id == 1) is enough for statistics.
crowd_u = anns_all[(anns_all.iscrowd == 1) & (anns_all.category_id == 1)].copy()
anns_all = anns_all[anns_all.iscrowd == 0].copy()

n_frames = imgs_all.groupby("split").size()
print("real objects:              ", anns_all.groupby("split").size().to_dict())
print("unique ignore regions:     ", crowd_u.groupby("split").size().to_dict())
print("share of frames with ignore:",
      (crowd_u.groupby("split")["image_id"].nunique() / n_frames).round(2).to_dict())

## 3. Frame resolution and video length

### 3.1 Resolution per split

In [ ]:
res = pd.concat([imgs_all[["split", "width", "height"]],
                 test_df[["width", "height"]].assign(split="test")])
res_table = (res.groupby(["width", "height", "split"]).size()
             .unstack("split", fill_value=0)[["train", "val", "test"]])
res_table

### 3.2 Frame rate and video length
`frame_id` step = 1 means that all consecutive frames are annotated.

In [ ]:
steps = imgs_all.sort_values(["video", "frame_id"]).groupby("video")["frame_id"].diff().dropna()
print("frame_id step inside videos:", steps.value_counts().to_dict())

lengths = pd.concat([
    imgs_all.groupby(["split", "video"]).size().rename("frames").reset_index(),
    test_df.groupby("clip").size().rename("frames").reset_index()
           .rename(columns={"clip": "video"}).assign(split="test"),
])
lengths.groupby("split")["frames"].describe()[["count", "mean", "min", "max"]].round(0)

How much of the validation set comes from each video:

In [ ]:
val_frames = imgs_all[imgs_all.split == "val"].groupby("video").size()
(val_frames / val_frames.sum()).round(2).sort_values(ascending=False).to_frame("share_of_val_frames")

## 4. Class distribution

In [ ]:
class_counts = anns_all.groupby(["category", "split"]).size().unstack("split", fill_value=0)
class_counts = class_counts.sort_values("train", ascending=False)
class_order = class_counts.index
class_share = (class_counts / class_counts.sum()).round(3)

ax = class_share.plot.bar(figsize=(8, 4), rot=30)
ax.set_title("Class share: train vs val")
ax.set_ylabel("fraction of objects")
plt.tight_layout(); plt.savefig(f"{FIGS}/class_share.png", dpi=120); plt.show()

print("car / bus ratio in train:", round(class_counts.loc["car", "train"] / class_counts.loc["bus", "train"], 1))
pd.concat([class_counts, class_share], axis=1, keys=["count", "share"])

## 5. Object sizes

### 5.1 Size distribution
Size = `sqrt(w*h)` in pixels of the original frame. Dashed lines are the COCO limits:
small < 32 px, medium 32–96 px, large > 96 px.

In [ ]:
anns_all["size_px"] = np.sqrt(anns_all.w * anns_all.h)

fig, ax = plt.subplots(figsize=(8, 4))
for split in ["train", "val"]:
    ax.hist(anns_all.loc[anns_all.split == split, "size_px"], bins=np.logspace(0, 3, 60),
            density=True, alpha=0.6, label=split)
for t in (32, 96):
    ax.axvline(t, color="k", ls="--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("sqrt(w*h), px"); ax.legend()
ax.set_title("Object size at original resolution")
plt.tight_layout(); plt.savefig(f"{FIGS}/object_size.png", dpi=120); plt.show()

anns_all.groupby("split")["size_px"].describe(percentiles=[.1, .5, .9])[["10%", "50%", "90%"]].round(1)

### 5.2 Small / medium / large share per class (COCO definition)

In [ ]:
anns_all["size_cat"] = pd.cut(anns_all.w * anns_all.h, [0, 32**2, 96**2, np.inf],
                              labels=["small", "medium", "large"])
size_table = pd.crosstab(anns_all.category, anns_all.size_cat, normalize="index").loc[class_order]
size_table.loc["ALL"] = anns_all.size_cat.value_counts(normalize=True).reindex(size_table.columns).values
size_table.round(2)

### 5.3 Effect of input resolution
Detectors resize every frame so that its longer side equals `imgsz`.
How many objects become tiny (< 8 px or < 16 px) after this resize?

In [ ]:
rows = []
for imgsz in [640, 960, 1280, 1920]:
    s = anns_all.size_px * imgsz / anns_all[["width", "height"]].max(axis=1)
    rows.append({"imgsz": imgsz,
                 "% objects < 8 px": round(100 * (s < 8).mean(), 1),
                 "% objects < 16 px": round(100 * (s < 16).mean(), 1),
                 "median size, px": round(s.median(), 1)})
pd.DataFrame(rows).set_index("imgsz")

## 6. Scene density
Number of real objects per frame (ignore boxes excluded).

In [ ]:
obj_per_frame = (anns_all.groupby(["split", "image_id"]).size()
                 .reindex(pd.MultiIndex.from_frame(imgs_all[["split", "id"]]), fill_value=0))
imgs_all["n_obj"] = obj_per_frame.values

imgs_all.groupby("split")["n_obj"].describe(percentiles=[.5, .9, .99])[["mean", "50%", "90%", "99%", "max"]].round(1)

Per-video statistics. The full table is saved to `<work>/video_stats.csv`; here only the extremes are shown.

In [ ]:
video_stats = (imgs_all.groupby(["split", "video"])
               .agg(frames=("id", "size"), width=("width", "first"), height=("height", "first"),
                    obj_mean=("n_obj", "mean"), obj_max=("n_obj", "max"))
               .round(1).sort_values("obj_mean", ascending=False))
video_stats.to_csv(f"{WORK}/video_stats.csv")

print("5 densest videos:");  display(video_stats.head(5))
print("5 sparsest videos:"); display(video_stats.tail(5))

In [ ]:
vs = video_stats.reset_index().sort_values("obj_mean")
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(vs)), vs["obj_mean"], color=vs["split"].map({"train": "tab:blue", "val": "tab:orange"}))
ax.set_yticks([]); ax.set_ylabel("videos (sorted)"); ax.set_xlabel("mean objects per frame")
ax.set_title("Scene density per video")
ax.legend(handles=[patches.Patch(color="tab:blue", label="train"),
                   patches.Patch(color="tab:orange", label="val")])
plt.tight_layout(); plt.savefig(f"{FIGS}/density_per_video.png", dpi=120); plt.show()

## 7. Ground-truth visualisation (conversion check)
Boxes are coloured by class, ignored regions are filled red. The title compares the number of
COCO objects with the number of lines in the YOLO label file for the same frame: they must be equal.

In [ ]:
def show_frame(split, video, frame_id, ax):
    """Draw GT boxes and ignored regions for one frame and check COCO vs YOLO label counts."""
    im = imgs_all[(imgs_all.split == split) & (imgs_all.video == video) &
                  (imgs_all.frame_id == frame_id)].iloc[0]
    ax.imshow(Image.open(f"{COMP}/{split}/{im.file_name}"))

    objs = anns_all[(anns_all.split == split) & (anns_all.image_id == im.id)]
    for r in objs.itertuples():
        ax.add_patch(patches.Rectangle((r.x, r.y), r.w, r.h, fill=False, lw=0.8,
                                       ec=COLORS[r.category_id - 1]))
    ign = crowd_u[(crowd_u.split == split) & (crowd_u.image_id == im.id)]
    for r in ign.itertuples():
        ax.add_patch(patches.Rectangle((r.x, r.y), r.w, r.h, color="red", alpha=0.35))

    with open(f"{COMP}/{split}/labels/{video}/{frame_id:07d}.txt") as f:
        n_yolo = sum(1 for _ in f)
    ax.set_title(f"{split} | {video} | {im.width}x{im.height}\n"
                 f"COCO objects={len(objs)}, YOLO lines={n_yolo}, ignore={len(ign)}", fontsize=10)
    ax.axis("off")

examples = [("train", "uav0000126_00001_v", 1),   # dense scene
            ("train", "uav0000145_00000_v", 1),   # sparse scene
            ("train", "uav0000013_00000_v", 1),   # many ignored regions
            ("val",   "uav0000268_05773_v", 1)]   # 4K validation video

fig, axes = plt.subplots(2, 2, figsize=(22, 13))
for ax, (s, v, f) in zip(axes.ravel(), examples):
    show_frame(s, v, f, ax)
handles = [patches.Patch(color=COLORS[i], label=n) for i, n in enumerate(CLASS_NAMES)]
handles.append(patches.Patch(color="red", alpha=0.35, label="ignored region"))
fig.legend(handles=handles, loc="lower center", ncol=9, fontsize=11)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(f"{FIGS}/gt_examples.png", dpi=110); plt.show()

In all examples the COCO and YOLO counts are the same, so both annotation formats describe the same
objects. Ignored regions cover crowds, far-away parking lots and traffic near the horizon. Real objects
are inside them but have no labels.

## 8. Summary

### Key observations

**Data and splits**
- 56 train videos (24,201 frames) and 7 val videos (2,846 frames). No video is in both splits,
  so the given split is already done by sequence and frames do not leak between train and val.
- All consecutive frames are annotated (`frame_id` step = 1). Neighbouring frames are almost the same.
- The validation set is small and uneven: one 4K video (`uav0000268_05773_v`) gives about one third
  of all val frames. 4K frames appear only in val, not in train or test.

**Classes**
- The 10 VisDrone classes were merged into 8: pedestrian + people → person,
  tricycle + awning-tricycle → tricycle.
- Strong imbalance: car and person are about 75% of all objects, bus is about 1%.
- Class shares differ between splits: person is more common in val, car in train.

**Ignored regions**
- Ignored regions and "others" are `iscrowd=1` boxes, copied for every class. They are not objects.
- 75% of train frames and 50% of val frames have at least one ignored region.
  They usually cover dense crowds and distant traffic, where objects exist but are not labelled.

**Object size and resolution**
- Objects are small: most are between ~15 and ~100 px at the original resolution.
  Person and motor are mostly small or medium; bus and truck are the largest classes.
- Train/val frames range from 1344×756 to 3840×2160.
- If frames are resized to 640 px, about half of the objects become smaller than 16 px.
  At 1280 px this drops to about 10%.
- The test set has 600 low-resolution frames (680×382 and 960×540) that do not appear in train or val.

**Density**
- On average about 46 objects per frame in train and 40 in val; up to 180 in one frame.
  Density changes a lot between videos (from ~5 to ~143 objects per frame on average).

### Decisions for the next notebooks

| Observation | Decision |
|---|---|
| Split is already by video | Use the given train/val split for all models, with the same metric (COCO mAP from `pycocotools`) |
| Consecutive frames are near-duplicates | Train on every 5th frame, validate on every 3rd frame |
| Objects are small | Use `imgsz=1280` as the default input size; 640 loses too many objects |
| Ignore boxes are copied per class | Remove `iscrowd=1` boxes from training targets; keep them for evaluation |
| Ignored regions contain unlabelled objects | Experiment: mask ignored regions in training images |
| Most objects are 15–100 px | Anchor sizes for RetinaNet should start below the default 32 px |
| Up to 180 objects per frame | Keep up to 500 detections per frame; use ≥ 300 queries for DETR |
| Low-resolution frames in test only | Experiment: random down-scaling augmentation |
| Strong class imbalance | Report per-class AP to see the effect on rare classes (bus, truck) |